# 多義語の語義を文脈から判定する（語義曖昧性解消）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_lexical_semantics/wsd_en.ipynb)

NLP クックブック 19-05 ／ 語の意味と意味の変化

`bank` が「銀行」なのか「土手」なのかを、文脈から自動で判定したい。WordNet の語義のどれに当たるかを選ぶ。

In [1]:
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk

sentences = ["I deposited my salary at the bank this morning.", "We had a picnic on the bank of the river.",
             "The pilot had to bank the plane sharply to the left.", "He caught a huge bass in the lake.",
             "She plays bass in a jazz band."]
targets = ["bank", "bank", "bank", "bass", "bass"]

# 1. Lesk アルゴリズム: 文脈の単語と、語義の定義文の単語の重なりが最も大きい語義を選ぶ
for sent, word in zip(sentences, targets):
    syn = lesk(nltk.word_tokenize(sent), word)
    print(f"{syn.name():22} {syn.definition()[:60]}\n    ← {sent}")

savings_bank.n.02      a container (usually with a slot in the top) for keeping mon
    ← I deposited my salary at the bank this morning.
bank.n.01              sloping land (especially the slope beside a body of water)
    ← We had a picnic on the bank of the river.
bank.n.07              a slope in the turn of a road or track; the outside is highe
    ← The pilot had to bank the plane sharply to the left.
bass.n.02              the lowest part in polyphonic music
    ← He caught a huge bass in the lake.
bass.n.02              the lowest part in polyphonic music
    ← She plays bass in a jazz band.


In [2]:
# 2. 定義文の埋め込みと文脈の埋め込みの類似度で選ぶ (単語の重なりがなくても意味が近ければ選べる)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

def wsd(sentence, word, pos=None):
    synsets = wn.synsets(word, pos=pos)
    glosses = [f"{word}: {s.definition()}. {' '.join(s.examples())}" for s in synsets]
    sims = model.encode(glosses, normalize_embeddings=True) @ model.encode(sentence, normalize_embeddings=True)
    best = sims.argsort()[::-1]
    return [(synsets[i].name(), round(float(sims[i]), 3), synsets[i].definition()) for i in best[:2]]

for sent, word in zip(sentences, targets):
    (name, score, definition), second = wsd(sent, word)
    print(f"{name:22} {score}  {definition[:60]}\n    ← {sent}   (次点: {second[0]} {second[1]})")

bank.v.03              0.348  do business with a bank or keep an account at a bank
    ← I deposited my salary at the bank this morning.   (次点: deposit.v.02 0.344)
bank.n.01              0.415  sloping land (especially the slope beside a body of water)
    ← We had a picnic on the bank of the river.   (次点: bank.n.03 0.255)
bank.v.01              0.712  tip laterally
    ← The pilot had to bank the plane sharply to the left.   (次点: bank.n.10 0.567)
freshwater_bass.n.01   0.491  any of various North American freshwater fish with lean fles
    ← He caught a huge bass in the lake.   (次点: bass.n.08 0.428)
bass.n.07              0.445  the member with the lowest range of a family of musical inst
    ← She plays bass in a jazz band.   (次点: bass.n.01 0.442)


- 単純な Lesk は、定義文が短いので単語がほとんど重ならず、精度が低い。埋め込みを使うと大きく改善する。
- 「常に最頻の語義（WordNet の 1 番目の語義）を選ぶ」というベースラインが非常に強いことで有名。評価するときは必ず比較する。
- WordNet の語義は細かすぎるという批判もある（`bank` の名詞だけで 10 個）。応用では、粗い語義にまとめて使うことが多い。
- 日本語 WordNet (`wn.synsets("首", lang="jpn")`) と多言語の埋め込みで同じことができるが、定義文が英語であることもあり、試した範囲では精度は低かった。日本語では次項の語義推定のほうが扱いやすい。